# Data Quality Checks

## Overview
Data quality is more than just testing for NULLs. A complete DQ framework covers: completeness (no missing required values), validity (values in acceptable ranges), uniqueness (no duplicates), consistency (cross-table integrity), timeliness (freshness), and accuracy (anomaly detection).

**DQ check categories:**

| Category | Examples |
|---|---|
| **Completeness** | NOT NULL checks, minimum row counts |
| **Validity** | Accepted values, range checks, format checks |
| **Uniqueness** | Duplicate detection (exact and near-duplicate) |
| **Consistency** | Referential integrity, cross-table business rules |
| **Timeliness** | Freshness checks, stale data detection |
| **Accuracy** | Statistical anomaly detection (z-score, IQR) |

---

In [1]:
import sqlite3, json
from datetime import date, timedelta

conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row

conn.executescript("""
-- Mixed dataset: insurance + healthcare + financial transactions
CREATE TABLE customers (
    customer_id  TEXT PRIMARY KEY,
    full_name    TEXT NOT NULL,
    email        TEXT,
    province     TEXT NOT NULL,
    segment      TEXT NOT NULL,   -- 'retail','commercial','group'
    created_at   TEXT NOT NULL
);

CREATE TABLE policies (
    policy_id    TEXT PRIMARY KEY,
    customer_id  TEXT NOT NULL REFERENCES customers(customer_id),
    product_line TEXT NOT NULL,   -- 'auto','home','life','health'
    premium      REAL NOT NULL,
    status       TEXT NOT NULL,   -- 'active','lapsed','cancelled'
    start_date   TEXT NOT NULL,
    end_date     TEXT
);

CREATE TABLE claims (
    claim_id     TEXT PRIMARY KEY,
    policy_id    TEXT NOT NULL REFERENCES policies(policy_id),
    claim_date   TEXT NOT NULL,
    amount       REAL NOT NULL,
    status       TEXT NOT NULL,   -- 'open','approved','denied','paid'
    approved_by  TEXT
);

CREATE TABLE lab_results (
    result_id    TEXT PRIMARY KEY,
    patient_id   TEXT NOT NULL,
    test_name    TEXT NOT NULL,
    result_value REAL,
    unit         TEXT,
    result_date  TEXT NOT NULL,
    flagged      INTEGER DEFAULT 0
);

-- Intentionally seeded with some data quality issues for demos
""")

# Clean customers
customers = [
    ('C001','Alice Nguyen',    'alice@email.com',  'ON','retail',    '2022-01-15'),
    ('C002','Bob Harrington',  'bob@email.com',    'BC','commercial','2022-03-01'),
    ('C003','Fatima Al-Rashid','fatima@email.com', 'QC','group',     '2022-06-10'),
    ('C004','James MacLeod',   'james@email.com',  'NS','retail',    '2023-01-20'),
    ('C005','Mei-Ling Chen',   'mei@email.com',    'AB','commercial','2023-04-05'),
    ('C006','David Park',      None,               'ON','retail',    '2024-01-01'),  # null email
    ('C007','Sandra Okafor',   'sandra@email.com', 'ON','retail',    '2024-02-15'),
]
conn.executemany("INSERT INTO customers VALUES (?,?,?,?,?,?)", customers)

# Policies — including some with data issues
policies = [
    ('POL-001','C001','auto',  1200.0,'active',   '2023-01-15','2024-01-15'),
    ('POL-002','C001','home',  1800.0,'active',   '2023-03-01','2024-03-01'),
    ('POL-003','C002','auto',  2400.0,'active',   '2022-06-01','2023-06-01'),
    ('POL-004','C003','life',   600.0,'lapsed',   '2022-09-01','2023-09-01'),
    ('POL-005','C004','auto',  1400.0,'active',   '2023-02-01','2024-02-01'),
    ('POL-006','C005','health',4800.0,'active',   '2023-05-01','2024-05-01'),
    ('POL-007','C006','home',  2100.0,'cancelled','2024-01-15','2024-06-15'),
    ('POL-008','C007','auto',  1100.0,'active',   '2024-03-01','2025-03-01'),
    ('POL-009','C002','home',  -500.0,'active',   '2024-01-01','2025-01-01'),  # negative premium (data issue)
    ('POL-010','C004','auto',  1400.0,'active',   '2023-02-01','2024-02-01'),  # duplicate of POL-005
]
conn.executemany("INSERT INTO policies VALUES (?,?,?,?,?,?,?)", policies)

# Claims
claims = [
    ('CLM-001','POL-001','2023-06-15',2200.0,'paid',    'Dr. Lee'),
    ('CLM-002','POL-003','2023-08-01', 500.0,'approved','Dr. Pham'),
    ('CLM-003','POL-005','2023-11-20',8500.0,'paid',    'Dr. Singh'),
    ('CLM-004','POL-006','2024-01-10', 350.0,'open',    None),
    ('CLM-005','POL-001','2024-02-28',1100.0,'denied',  'Dr. Lee'),
    ('CLM-006','POL-008','2024-04-15', 750.0,'approved','Dr. Pham'),
    ('CLM-007','POL-003','2024-05-01',99999.0,'open',   None),  # suspiciously large
]
conn.executemany("INSERT INTO claims VALUES (?,?,?,?,?,?)", claims)

# Lab results — with some quality issues
labs = [
    ('LAB-001','P001','HbA1c',    7.2,'%',        '2023-10-01',1),
    ('LAB-002','P001','eGFR',    68.0,'mL/min',   '2023-10-01',0),
    ('LAB-003','P002','HbA1c',    8.4,'%',        '2024-01-10',1),
    ('LAB-004','P002','Creatinine',115.0,'umol/L','2024-01-10',1),
    ('LAB-005','P003','HbA1c',    None,'%',        '2024-02-15',0),  # null value
    ('LAB-006','P001','HbA1c',    7.2,'%',        '2023-10-01',1),  # duplicate of LAB-001
    ('LAB-007','P004','eGFR',   -5.0,'mL/min',   '2024-03-01',0),  # impossible negative
    ('LAB-008','P003','HbA1c',    9.1,'%',        '2024-02-15',1),
]
conn.executemany("INSERT INTO lab_results VALUES (?,?,?,?,?,?,?)", labs)
conn.commit()

print("Testing dataset loaded:")
for tbl in ['customers','policies','claims','lab_results']:
    n = conn.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"  {tbl:<15s}: {n} rows")
print()
print("Known data quality issues seeded:")
issues = [
    ("customers",    "C006 has NULL email"),
    ("policies",     "POL-009 has negative premium (-500)"),
    ("policies",     "POL-010 is a near-duplicate of POL-005"),
    ("claims",       "CLM-007 has suspiciously large amount (99999)"),
    ("lab_results",  "LAB-005 has NULL result_value"),
    ("lab_results",  "LAB-006 is an exact duplicate of LAB-001"),
    ("lab_results",  "LAB-007 has impossible negative eGFR (-5)"),
]
for tbl, issue in issues:
    print(f"  [{tbl}] {issue}")


Testing dataset loaded:
  customers      : 7 rows
  policies       : 10 rows
  claims         : 7 rows
  lab_results    : 8 rows

Known data quality issues seeded:
  [customers] C006 has NULL email
  [policies] POL-009 has negative premium (-500)
  [policies] POL-010 is a near-duplicate of POL-005
  [claims] CLM-007 has suspiciously large amount (99999)
  [lab_results] LAB-005 has NULL result_value
  [lab_results] LAB-006 is an exact duplicate of LAB-001
  [lab_results] LAB-007 has impossible negative eGFR (-5)


---
## DQ dashboard: all checks in one view

In [2]:
print("=== Data quality dashboard: all checks in one query ===")
print()

checks = [
    ("null_premiums",           "SELECT COUNT(*) FROM policies WHERE premium IS NULL"),
    ("negative_premiums",       "SELECT COUNT(*) FROM policies WHERE premium < 0"),
    ("invalid_policy_status",   "SELECT COUNT(*) FROM policies WHERE status NOT IN ('active','lapsed','cancelled')"),
    ("orphan_policies",         "SELECT COUNT(*) FROM policies p LEFT JOIN customers c ON c.customer_id=p.customer_id WHERE c.customer_id IS NULL"),
    ("duplicate_lab_results",   "SELECT COUNT(*) FROM (SELECT patient_id,test_name,result_date FROM lab_results GROUP BY patient_id,test_name,result_date HAVING COUNT(*)>1)"),
    ("null_lab_values",         "SELECT COUNT(*) FROM lab_results WHERE result_value IS NULL"),
    ("negative_egfr",           "SELECT COUNT(*) FROM lab_results WHERE test_name='eGFR' AND result_value < 0"),
    ("extreme_claims",          "SELECT COUNT(*) FROM claims WHERE amount > 50000"),
    ("null_customer_names",     "SELECT COUNT(*) FROM customers WHERE full_name IS NULL OR TRIM(full_name)=''"),
    ("invalid_claim_status",    "SELECT COUNT(*) FROM claims WHERE status NOT IN ('open','approved','denied','paid')"),
]

print(f"  {'check_name':<28s}  {'violations':>10s}  {'status'}")
print("  " + "-"*52)
total_violations = 0
for name, sql in checks:
    n = conn.execute(sql).fetchone()[0]
    total_violations += n
    status = "PASS ✓" if n == 0 else "FAIL ✗"
    print(f"  {name:<28s}  {n:>10d}  {status}")

print(f"\n  Total violations found: {total_violations}")
print()
print("Building this as a PostgreSQL view:")
print("""
  CREATE OR REPLACE VIEW v_data_quality_dashboard AS
  SELECT 'null_premiums'       AS check_name, COUNT(*) AS violations
  FROM   policies WHERE premium IS NULL
  UNION ALL
  SELECT 'negative_premiums',  COUNT(*)
  FROM   policies WHERE premium < 0
  UNION ALL
  SELECT 'invalid_policy_status', COUNT(*)
  FROM   policies WHERE status NOT IN ('active','lapsed','cancelled')
  UNION ALL
  SELECT 'orphan_policies',    COUNT(*)
  FROM   policies p
  LEFT JOIN customers c ON c.customer_id = p.customer_id
  WHERE  c.customer_id IS NULL
  UNION ALL
  SELECT 'extreme_claims',     COUNT(*)
  FROM   claims WHERE amount > 50000;

  -- Query it simply:
  SELECT * FROM v_data_quality_dashboard WHERE violations > 0;
""")


=== Data quality dashboard: all checks in one query ===

  check_name                    violations  status
  ----------------------------------------------------
  null_premiums                          0  PASS ✓
  negative_premiums                      1  FAIL ✗
  invalid_policy_status                  0  PASS ✓
  orphan_policies                        0  PASS ✓
  duplicate_lab_results                  2  FAIL ✗
  null_lab_values                        1  FAIL ✗
  negative_egfr                          1  FAIL ✗
  extreme_claims                         1  FAIL ✗
  null_customer_names                    0  PASS ✓
  invalid_claim_status                   0  PASS ✓

  Total violations found: 6

Building this as a PostgreSQL view:

  CREATE OR REPLACE VIEW v_data_quality_dashboard AS
  SELECT 'null_premiums'       AS check_name, COUNT(*) AS violations
  FROM   policies WHERE premium IS NULL
  UNION ALL
  SELECT 'negative_premiums',  COUNT(*)
  FROM   policies WHERE premium < 0
  UNION AL

---
## Anomaly detection with Z-score and IQR

In [3]:
print("=== Anomaly detection with SQL ===")
print()

# Z-score based anomaly detection on claim amounts
rows = conn.execute("""
    WITH stats AS (
        SELECT AVG(amount)                          AS mean_amount,
               -- SQLite lacks STDDEV; use variance formula
               SQRT(AVG(amount*amount) - AVG(amount)*AVG(amount)) AS stddev_amount
        FROM   claims
        WHERE  amount < 50000   -- exclude known outliers from baseline
    )
    SELECT c.claim_id,
           c.amount,
           s.mean_amount,
           s.stddev_amount,
           ROUND((c.amount - s.mean_amount) / NULLIF(s.stddev_amount, 0), 2) AS z_score
    FROM   claims c, stats s
    ORDER  BY z_score DESC
""").fetchall()

print("Claim amount Z-scores (outlier detection):")
print(f"  {'claim_id':<10s}  {'amount':>10s}  {'mean':>8s}  {'stddev':>8s}  {'z_score':>8s}  flag")
print("  " + "-"*55)
for r in rows:
    flag = '⚠ OUTLIER' if abs(r['z_score'] or 0) > 2 else ''
    print(f"  {r['claim_id']:<10s}  ${r['amount']:>9,.0f}  ${r['mean_amount']:>7,.0f}  "
          f"${r['stddev_amount']:>7,.0f}  {r['z_score']:>8.2f}  {flag}")

print()
# Interquartile range (IQR) method
rows2 = conn.execute("""
    WITH ordered AS (
        SELECT amount,
               ROW_NUMBER() OVER (ORDER BY amount) AS rn,
               COUNT(*) OVER () AS total
        FROM   claims WHERE amount < 50000
    ),
    quartiles AS (
        SELECT
            AVG(CASE WHEN rn BETWEEN total*0.24 AND total*0.26 THEN amount END) AS q1,
            AVG(CASE WHEN rn BETWEEN total*0.74 AND total*0.76 THEN amount END) AS q3
        FROM ordered
    )
    SELECT c.claim_id, c.amount,
           q.q1, q.q3,
           q.q3 - q.q1 AS iqr,
           q.q1 - 1.5*(q.q3-q.q1) AS lower_fence,
           q.q3 + 1.5*(q.q3-q.q1) AS upper_fence
    FROM   claims c, quartiles q
    WHERE  c.amount > q.q3 + 1.5*(q.q3-q.q1)
       OR  c.amount < q.q1 - 1.5*(q.q3-q.q1)
""").fetchall()

print("IQR-based outliers (amount outside 1.5×IQR fences):")
for r in rows2:
    print(f"  {r['claim_id']}  amount=${r['amount']:,.0f}  "
          f"fence=[${r['lower_fence']:,.0f}, ${r['upper_fence']:,.0f}]")

print()
print("PostgreSQL: use built-in STDDEV_POP and PERCENTILE_CONT:")
print("""
  WITH stats AS (
      SELECT AVG(amount)      AS mean_val,
             STDDEV_POP(amount) AS std_val,
             PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY amount) AS q1,
             PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY amount) AS q3
      FROM   claims WHERE amount < 50000
  )
  SELECT claim_id, amount,
         (amount - mean_val) / NULLIF(std_val, 0) AS z_score,
         CASE WHEN amount > q3 + 1.5*(q3-q1) THEN 'high_outlier'
              WHEN amount < q1 - 1.5*(q3-q1) THEN 'low_outlier'
              ELSE 'normal' END AS iqr_flag
  FROM   claims, stats
  ORDER  BY z_score DESC;
""")


=== Anomaly detection with SQL ===

Claim amount Z-scores (outlier detection):
  claim_id        amount      mean    stddev   z_score  flag
  -------------------------------------------------------
  CLM-007     $   99,999  $  2,233  $  2,867     34.10  ⚠ OUTLIER
  CLM-003     $    8,500  $  2,233  $  2,867      2.19  ⚠ OUTLIER
  CLM-001     $    2,200  $  2,233  $  2,867     -0.01  
  CLM-005     $    1,100  $  2,233  $  2,867     -0.40  
  CLM-006     $      750  $  2,233  $  2,867     -0.52  
  CLM-002     $      500  $  2,233  $  2,867     -0.60  
  CLM-004     $      350  $  2,233  $  2,867     -0.66  

IQR-based outliers (amount outside 1.5×IQR fences):

PostgreSQL: use built-in STDDEV_POP and PERCENTILE_CONT:

  WITH stats AS (
      SELECT AVG(amount)      AS mean_val,
             STDDEV_POP(amount) AS std_val,
             PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY amount) AS q1,
             PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY amount) AS q3
      FROM   claims WHE

---
## Referential integrity and freshness

In [4]:
import datetime

print("=== Referential integrity and freshness checks ===")
print()

# Full referential integrity audit
print("Referential integrity audit:")
ri_checks = [
    ("policies → customers",
     "SELECT p.policy_id, p.customer_id FROM policies p LEFT JOIN customers c ON c.customer_id=p.customer_id WHERE c.customer_id IS NULL"),
    ("claims → policies",
     "SELECT cl.claim_id, cl.policy_id FROM claims cl LEFT JOIN policies p ON p.policy_id=cl.policy_id WHERE p.policy_id IS NULL"),
]
for check_name, sql in ri_checks:
    rows = conn.execute(sql).fetchall()
    status = "PASS ✓" if not rows else f"FAIL ✗  ({len(rows)} orphans)"
    print(f"  [{status}] {check_name}")

print()
# Cross-table consistency
print("Cross-table consistency checks:")
consistency = [
    ("claim_amount_vs_coverage",
     """SELECT cl.claim_id, cl.amount, p.premium
        FROM   claims cl JOIN policies p ON p.policy_id = cl.policy_id
        WHERE  cl.amount > p.premium * 20""",
     "Claim amount > 20x annual premium (implausible)"),
    ("lapsed_policy_has_active_claims",
     """SELECT cl.claim_id, p.status AS policy_status
        FROM   claims cl
        JOIN   policies p ON p.policy_id = cl.policy_id
        WHERE  p.status IN ('lapsed','cancelled')
          AND  cl.status = 'open'""",
     "Open claim on lapsed/cancelled policy"),
]
for check_name, sql, desc in consistency:
    rows = conn.execute(sql).fetchall()
    status = "PASS ✓" if not rows else f"FAIL ✗  ({len(rows)} rows)"
    print(f"  [{status}] {check_name}: {desc}")

print()
print("Freshness monitoring pattern (track load timestamps):")
print("""
  CREATE TABLE IF NOT EXISTS load_audit (
      table_name    TEXT PRIMARY KEY,
      last_loaded   TIMESTAMPTZ,
      rows_loaded   INTEGER,
      expected_freq TEXT   -- 'daily', 'hourly', etc.
  );

  -- After each ETL run:
  INSERT INTO load_audit VALUES ('policies', NOW(), 1250, 'daily')
  ON CONFLICT (table_name) DO UPDATE
      SET last_loaded = EXCLUDED.last_loaded,
          rows_loaded = EXCLUDED.rows_loaded;

  -- Freshness alert query:
  SELECT table_name, last_loaded, expected_freq,
         NOW() - last_loaded AS time_since_load,
         CASE expected_freq
             WHEN 'hourly' THEN last_loaded < NOW() - INTERVAL '2 hours'
             WHEN 'daily'  THEN last_loaded < NOW() - INTERVAL '25 hours'
         END AS is_stale
  FROM   load_audit
  WHERE  CASE expected_freq
             WHEN 'hourly' THEN last_loaded < NOW() - INTERVAL '2 hours'
             WHEN 'daily'  THEN last_loaded < NOW() - INTERVAL '25 hours'
         END = true;
""")


=== Referential integrity and freshness checks ===

Referential integrity audit:
  [PASS ✓] policies → customers
  [PASS ✓] claims → policies

Cross-table consistency checks:
  [FAIL ✗  (1 rows)] claim_amount_vs_coverage: Claim amount > 20x annual premium (implausible)
  [PASS ✓] lapsed_policy_has_active_claims: Open claim on lapsed/cancelled policy

Freshness monitoring pattern (track load timestamps):

  CREATE TABLE IF NOT EXISTS load_audit (
      table_name    TEXT PRIMARY KEY,
      last_loaded   TIMESTAMPTZ,
      rows_loaded   INTEGER,
      expected_freq TEXT   -- 'daily', 'hourly', etc.
  );

  -- After each ETL run:
  INSERT INTO load_audit VALUES ('policies', NOW(), 1250, 'daily')
  ON CONFLICT (table_name) DO UPDATE
      SET last_loaded = EXCLUDED.last_loaded,
          rows_loaded = EXCLUDED.rows_loaded;

  -- Freshness alert query:
  SELECT table_name, last_loaded, expected_freq,
         NOW() - last_loaded AS time_since_load,
         CASE expected_freq
             W

---
## Common Pitfalls

**1. Running anomaly detection on the full dataset including known outliers**
Z-score and IQR methods assume approximate normality. If the dataset contains a few known extreme values (e.g., CLM-007 at $99,999), they inflate the mean and standard deviation, making the method less sensitive to true anomalies. Filter known outliers from the baseline statistics, then flag against the cleaned distribution.

**2. Treating freshness checks as optional**
A data warehouse with stale data is often worse than no warehouse — users make decisions on outdated numbers without realising it. Freshness checks should be first-class tests that fail the pipeline loudly, not optional warnings.

**3. Referential integrity check using INNER JOIN instead of LEFT JOIN**
`INNER JOIN customers ON ...` silently drops orphan policy rows — the query succeeds but you never see the orphans. Always use `LEFT JOIN ... WHERE right.key IS NULL` to surface records that fail the integrity check.

**4. DQ dashboard as a view that consumers rely on for production**
A `UNION ALL` DQ dashboard view runs all its subqueries on every select, which is expensive on large tables. Materialise it as a scheduled table or run it as a batch job, not as a live view queried by dashboards.

**5. Z-score thresholds that are too tight or too loose**
A z-score threshold of 1.5 will flag 13% of a normal distribution as anomalies. A threshold of 4.0 may miss real fraud. Choose thresholds based on the business context: claims data warrants stricter thresholds (potential fraud) than lab results (physiological variation is expected).

**6. Not logging which checks failed and when**
Running DQ checks without recording results means you cannot track whether data quality is improving or degrading over time. Log each check result with a timestamp into a `dq_results` table and build a trend dashboard on top of it.


---
*sql_methods_library - Samantha McGarrigle*